# Week 5 Assignment


## Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

### Answer

Traditional **MapReduce** processes data in two phases: **Map** and **Reduce**. After each phase, it stores the intermediate data on **disk**, which increases **Disk I/O (Input/Output)** and slows down processing.

### Limitations of MapReduce

- **High Disk I/O:** Intermediate data is repeatedly written to and read from disk.
- **Slow Performance:** Disk access is much slower than memory (RAM).
- **Not Suitable for Iterative Processing:** Machine Learning algorithms process the same data multiple times, making MapReduce inefficient.
- **High Latency:** Mainly designed for **Batch Processing** and not suitable for real-time applications.
- **Complex Programming Model:** Developers need to write separate **Map** and **Reduce** functions.

### Why Spark is Preferred

- Uses **In-Memory Computing** to store intermediate data in **RAM**.
- Reduces **Disk I/O**, resulting in faster processing.
- Supports **Batch Processing**, **Real-Time Processing**, **Machine Learning**, and **Streaming**.
- Provides simple APIs in **Python, SQL, Scala, and Java**.

### Example

Suppose a company wants to calculate the total sales from a **1 TB** dataset.

**MapReduce:**
1. Read data from disk.
2. Process data.
3. Write intermediate results to disk.
4. Read them again for the next stage.

**Spark:**
1. Read data once.
2. Store intermediate data in **RAM**.
3. Process data.
4. Write only the final result to disk.

**Result:** Spark is much faster because it minimizes **Disk I/O**.



## Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

### Answer

**In-Memory Computing** is a feature of Spark that stores intermediate data in **RAM (Memory)** instead of repeatedly reading and writing it to disk.

An **Iterative Machine Learning Algorithm** performs the same operation multiple times on the same dataset until it reaches the desired result.

### MapReduce Approach

For every iteration:
1. Read data from disk.
2. Process the data.
3. Write the result back to disk.

This repeated **Disk I/O** makes processing slow.

### Spark Approach

1. Read the data only once.
2. Store the data in **RAM**.
3. Reuse the same data from memory for every iteration.
4. Write the final result to disk after processing is complete.

### Example

Suppose a Machine Learning model needs to train on customer data **50 times**.

**MapReduce:**
- Reads and writes the dataset to disk in every iteration.

**Spark:**
- Reads the dataset once.
- Stores it in **RAM**.
- Reuses it for all **50 iterations**.

**Result:** Spark significantly reduces **Disk I/O**, making iterative Machine Learning algorithms much faster.

## Spark Session Setup

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os
import sys
print(pyspark.__version__) #verify version of spark

4.1.2


In [2]:
# Create Spark Session

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .master("local[*]") \
    .getOrCreate()

### load Superstore.csv file 

In [3]:
df =  spark.read.csv("Superstore.csv", header = True, inferSchema=True)
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 


## Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: `user_id` and `transaction_date`.

### Answer

The **dropDuplicates()** method is used to remove duplicate rows from a DataFrame. Here, we remove duplicate records using **Customer ID** and **Order Date** from the Superstore dataset.

### Code

In [4]:
# Remove duplicate rows based on Customer ID and Order Date
df.count()
df = df.dropDuplicates(["Customer ID", "Order Date"])
print("DataCount after removing duplicates:")
df.count()

DataCount after removing duplicates:


4992

In [5]:
df = df.withColumn("Sales", expr("try_cast (Sales as double)"))
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



## Q4: Given a DataFrame `df_sales`, write a query to filter for rows where the **Region** is **'West'** and then group by **Category** to find the average **Sales**.

### Answer

The following code filters the records where the **Region** is **'West'** and then groups the data by **Category** to calculate the average **Sales** for each product category.

### Explanation

- **filter()** selects only the rows where the **Region** is **'West'**.
- **groupBy("Category","Region")** groups the records by product category and Region.
- **avg("Sales")** calculates the average sales for each category.
- **alias("Average Sales")** renames the output column for better readability.

In [10]:
df.filter(col("Region") == "West").groupBy("Category","Region").avg("Sales").alias("Avg Sale").show()

+---------------+------+------------------+
|       Category|Region|        avg(Sales)|
+---------------+------+------------------+
|Office Supplies|  West|102.14556873614184|
|      Furniture|  West| 343.2295197368419|
|     Technology|  West|383.11840225563935|
+---------------+------+------------------+



## Q5: What is the difference between `.na.drop()` and `.na.fill()`? Provide a code example of filling null values in a `status` column with the string **'Unknown'**.

### Answer

| `.na.drop()` | `.na.fill()` |
|--------------|--------------|
| Removes rows that contain null values. | Replaces null values with a specified value. |
| Used when incomplete records are not required. | Used when we want to keep all records by filling missing values. |

### Example

The following example replaces all null values in the **status** column with **"Unknown"**.

In [3]:
# Create a sample DataFrame with 20 records
df_status = spark.createDataFrame([
    (1, "Completed"),
    (2, None),
    (3, "Pending"),
    (4, "Completed"),
    (5, None),
    (6, "Cancelled"),
    (7, "Pending"),
    (8, None),
    (9, "Completed"),
    (10, None),
    (11, "Pending"),
    (12, "Completed"),
    (13, None),
    (14, "Cancelled"),
    (15, "Completed"),
    (16, None),
    (17, "Pending"),
    (18, "Completed"),
    (19, None),
    (20, "Cancelled")
], ["id", "status"])

print("Original DataFrame:")
df_status.show()

Original DataFrame:
+---+---------+
| id|   status|
+---+---------+
|  1|Completed|
|  2|     NULL|
|  3|  Pending|
|  4|Completed|
|  5|     NULL|
|  6|Cancelled|
|  7|  Pending|
|  8|     NULL|
|  9|Completed|
| 10|     NULL|
| 11|  Pending|
| 12|Completed|
| 13|     NULL|
| 14|Cancelled|
| 15|Completed|
| 16|     NULL|
| 17|  Pending|
| 18|Completed|
| 19|     NULL|
| 20|Cancelled|
+---+---------+



In [6]:
# Fill null values in the 'status' column with 'Unknown'
df_filled = df_status.na.fill({"status": "Unknown"})

print("DataFrame after filling null values:")
df_filled.show()

DataFrame after filling null values:
+---+---------+
| id|   status|
+---+---------+
|  1|Completed|
|  2|  Unknown|
|  3|  Pending|
|  4|Completed|
|  5|  Unknown|
|  6|Cancelled|
|  7|  Pending|
|  8|  Unknown|
|  9|Completed|
| 10|  Unknown|
| 11|  Pending|
| 12|Completed|
| 13|  Unknown|
| 14|Cancelled|
| 15|Completed|
| 16|  Unknown|
| 17|  Pending|
| 18|Completed|
| 19|  Unknown|
| 20|Cancelled|
+---+---------+



### Explanation

- **`.na.drop()`** removes rows that contain one or more null values.
- **`.na.fill()`** replaces null values with a specified value instead of removing the rows.
- In this example, all null values in the **status** column are replaced with **"Unknown"**, ensuring that no records are lost.

## Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

### Answer

The following query groups the data by **City**, counts the number of records for each city, and displays only those cities whose record count is greater than **100**.

In [15]:
df.groupBy("City").agg(count("*").alias("Total Records")).filter(col("Total Records") > 100).show()

+-------------+-------------+
|         City|Total Records|
+-------------+-------------+
| Philadelphia|          264|
|  Los Angeles|          383|
|San Francisco|          263|
|     Columbus|          111|
|      Chicago|          171|
|      Seattle|          211|
|New York City|          446|
|      Houston|          188|
+-------------+-------------+



### Explanation

- **groupBy("City")** groups all records based on the **City** column.
- **count("*")** counts the total number of records in each city.
- **alias("Total Records")** renames the count column for better readability.
- **filter()** displays only those cities where the total number of records is greater than **100**.

## Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

### Answer

Spark **DataFrames are immutable**, which means they **cannot be modified after they are created**. Any data cleaning operation creates a **new DataFrame** instead of changing the original one.

### Effect on Data Cleaning

- Operations like **dropping columns**, **renaming columns**, **filtering rows**, or **adding new columns** do not modify the existing DataFrame.
- Each operation returns a **new DataFrame** with the required changes.
- The original DataFrame remains unchanged unless the new DataFrame is assigned to a variable.

### Example

```python
# Drop a column
df_new = df.drop("Postal Code")

# Rename a column
df_new = df.withColumnRenamed("Customer Name", "Customer_Name")
```

### Explanation

- `drop()` creates a new DataFrame without the specified column.
- `withColumnRenamed()` creates a new DataFrame with the renamed column.
- The original DataFrame `df` remains unchanged.

### Advantages of Immutability

- Prevents accidental modification of data.
- Makes Spark more **fault-tolerant**.
- Helps Spark optimize execution using **Lazy Evaluation** and **DAG (Directed Acyclic Graph)**.

## Q8: Write a Spark command to filter a dataset for rows where the **age** is between **18 and 30** (inclusive) and the **subscription** is **'Premium'**.

### Answer

The **Superstore** dataset does not contain the columns **age** and **subscription**. Therefore, a sample DataFrame has been created to demonstrate the solution.

In [18]:
from pyspark.sql import Row

# Sample Data
data = [
    Row(id=1, name="Raj", age=22, subscription="Premium"),
    Row(id=2, name="Aman", age=17, subscription="Basic"),
    Row(id=3, name="Priya", age=28, subscription="Premium"),
    Row(id=4, name="Neha", age=30, subscription="Premium"),
    Row(id=5, name="Rahul", age=35, subscription="Premium"),
    Row(id=6, name="Simran", age=25, subscription="Basic"),
    Row(id=7, name="Karan", age=18, subscription="Premium"),
    Row(id=8, name="Riya", age=29, subscription="Premium"),
    Row(id=9, name="Vikas", age=31, subscription="Basic"),
    Row(id=10, name="Anjali", age=24, subscription="Premium")
]

# Create DataFrame
df_users = spark.createDataFrame(data)

print("Original DataFrame:")
df_users.show()

Original DataFrame:
+---+------+---+------------+
| id|  name|age|subscription|
+---+------+---+------------+
|  1|   Raj| 22|     Premium|
|  2|  Aman| 17|       Basic|
|  3| Priya| 28|     Premium|
|  4|  Neha| 30|     Premium|
|  5| Rahul| 35|     Premium|
|  6|Simran| 25|       Basic|
|  7| Karan| 18|     Premium|
|  8|  Riya| 29|     Premium|
|  9| Vikas| 31|       Basic|
| 10|Anjali| 24|     Premium|
+---+------+---+------------+



In [19]:
# Filter rows where age is between 18 and 30 (inclusive)
# and subscription is 'Premium'

filtered_df = df_users.filter(
    (df_users.age.between(18, 30)) &
    (df_users.subscription == "Premium")
)

print("Filtered DataFrame:")
filtered_df.show()

Filtered DataFrame:
+---+------+---+------------+
| id|  name|age|subscription|
+---+------+---+------------+
|  1|   Raj| 22|     Premium|
|  3| Priya| 28|     Premium|
|  4|  Neha| 30|     Premium|
|  7| Karan| 18|     Premium|
|  8|  Riya| 29|     Premium|
| 10|Anjali| 24|     Premium|
+---+------+---+------------+



### Explanation

- **between(18, 30)** filters records where the age is between **18** and **30**, including both values.
- **&** is the logical **AND** operator in PySpark.
- The second condition checks whether the **subscription** is **'Premium'**.
- Only rows satisfying **both conditions** are returned.

## Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like `sum()` or `avg()`?

### Answer

It is important to handle **null values** before performing mathematical aggregations because null values can affect the accuracy and reliability of the results.

### Reasons

- **Improves Accuracy:** Missing values may lead to incorrect calculations if they are not handled properly.
- **Ensures Data Quality:** Cleaning null values produces more reliable and meaningful analysis.
- **Prevents Errors:** Some operations or business rules may fail if null values are present.
- **Provides Better Insights:** Replacing or removing null values helps generate more accurate reports and analytics.

### Example

Suppose the **Sales** column contains the following values:

| Sales |
|-------:|
| 100 |
| 200 |
| NULL |
| 300 |

If the null value is not handled, the calculated average or other statistics may not represent the actual data correctly.

A common approach is to:
- Remove rows with null values using **`.na.drop()`**, or
- Replace null values using **`.na.fill()`**, depending on the business requirement.

### Conclusion

Handling null values before using functions like **`sum()`**, **`avg()`**, or **`count()`** improves data quality and ensures accurate aggregation results.

## Q10: Write the code to revise a column named `raw_timestamp` by casting it to a `TimestampType` and renaming it to `event_time`.

### Answer

The **Superstore** dataset does not contain a **raw_timestamp** column. Therefore, a sample DataFrame has been created to demonstrate the solution.

The following code converts the **raw_timestamp** column from **StringType** to **TimestampType** and renames it to **event_time**.

In [22]:
from pyspark.sql.types import TimestampType

# Sample Data
data = [
    Row(id=1, raw_timestamp="2025-07-01 10:30:00"),
    Row(id=2, raw_timestamp="2025-07-02 14:45:10"),
    Row(id=3, raw_timestamp="2025-07-03 09:15:25"),
    Row(id=4, raw_timestamp="2025-07-04 18:20:40"),
    Row(id=5, raw_timestamp="2025-07-05 12:10:55")
]

# Create DataFrame
df_time = spark.createDataFrame(data)

print("Original DataFrame:")
df_time.show(truncate=False)

Original DataFrame:
+---+-------------------+
|id |raw_timestamp      |
+---+-------------------+
|1  |2025-07-01 10:30:00|
|2  |2025-07-02 14:45:10|
|3  |2025-07-03 09:15:25|
|4  |2025-07-04 18:20:40|
|5  |2025-07-05 12:10:55|
+---+-------------------+



In [23]:
# Cast raw_timestamp to TimestampType and rename it to event_time

df_time = df_time.withColumn(
    "raw_timestamp",
    col("raw_timestamp").cast(TimestampType())
).withColumnRenamed(
    "raw_timestamp",
    "event_time"
)

print("Updated DataFrame:")
df_time.show(truncate=False)

Updated DataFrame:
+---+-------------------+
|id |event_time         |
+---+-------------------+
|1  |2025-07-01 10:30:00|
|2  |2025-07-02 14:45:10|
|3  |2025-07-03 09:15:25|
|4  |2025-07-04 18:20:40|
|5  |2025-07-05 12:10:55|
+---+-------------------+



### Explanation

- **cast(TimestampType())** converts the **raw_timestamp** column from a string to **TimestampType**.
- **withColumn()** updates the data type of the existing column.
- **withColumnRenamed()** renames the column from **raw_timestamp** to **event_time**.
- The final DataFrame contains the new **event_time** column with the **TimestampType** data type.

## Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

### Answer

**Shuffle** is the process of **moving data between different partitions (executors)** so that records with the same key are grouped together.

When we perform operations like **groupBy()**, **join()**, or **distinct()**, Spark redistributes the data across the cluster. This data movement is called **Shuffle**.

### Why is it a Wide Transformation?

A **Wide Transformation** is an operation where data from **multiple partitions** is required to produce the final result.

During a **groupBy()**, records with the same key may be stored in different partitions. Spark performs a **Shuffle** to bring all matching records into the same partition before applying the aggregation.

### Example

Suppose the dataset contains the following records:

| City | Sales |
|------|------:|
| Delhi | 100 |
| Mumbai | 200 |
| Delhi | 150 |
| Mumbai | 300 |

When we execute:

```python
df.groupBy("City").sum("Sales")
```

Spark first **shuffles** the data so that all **Delhi** records are moved to one partition and all **Mumbai** records are moved to another partition. Then it calculates the total sales for each city.

### Why is Shuffle Expensive?

- It transfers data across the network.
- It increases **Disk I/O** and **Network I/O**.
- It takes more time than transformations that process data within the same partition.

### Conclusion

Shuffle is considered a **Wide Transformation** because it requires data to move between multiple partitions before the operation can be completed.

## Q12: Write a code snippet that identifies and removes rows where the `email` column contains null values OR the `username` is an empty string.

### Answer

The **Superstore** dataset does not contain the **email** and **username** columns. Therefore, a sample DataFrame has been created to demonstrate the solution.

The following code removes rows where:
- The **email** column contains **null** values, or
- The **username** column is an empty string (`""`).

In [25]:
# Sample Data
data = [
    Row(id=1, username="raj123", email="raj@gmail.com"),
    Row(id=2, username="", email="aman@gmail.com"),
    Row(id=3, username="priya", email=None),
    Row(id=4, username="neha", email="neha@gmail.com"),
    Row(id=5, username="", email=None),
    Row(id=6, username="rahul", email="rahul@gmail.com"),
    Row(id=7, username="simran", email=None),
    Row(id=8, username="karan", email="karan@gmail.com"),
    Row(id=9, username="", email="riya@gmail.com"),
    Row(id=10, username="vikas", email="vikas@gmail.com")
]

# Create DataFrame
df_users = spark.createDataFrame(data)

print("Original DataFrame:")
df_users.show()

Original DataFrame:
+---+--------+---------------+
| id|username|          email|
+---+--------+---------------+
|  1|  raj123|  raj@gmail.com|
|  2|        | aman@gmail.com|
|  3|   priya|           NULL|
|  4|    neha| neha@gmail.com|
|  5|        |           NULL|
|  6|   rahul|rahul@gmail.com|
|  7|  simran|           NULL|
|  8|   karan|karan@gmail.com|
|  9|        | riya@gmail.com|
| 10|   vikas|vikas@gmail.com|
+---+--------+---------------+



In [26]:
# Remove rows where email is NULL or username is empty

clean_df = df_users.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

print("Cleaned DataFrame:")
clean_df.show()

Cleaned DataFrame:
+---+--------+---------------+
| id|username|          email|
+---+--------+---------------+
|  1|  raj123|  raj@gmail.com|
|  4|    neha| neha@gmail.com|
|  6|   rahul|rahul@gmail.com|
|  8|   karan|karan@gmail.com|
| 10|   vikas|vikas@gmail.com|
+---+--------+---------------+



### Explanation

- **isNotNull()** keeps only the rows where the **email** column is not null.
- **(col("username") != "")** keeps only the rows where the **username** is not an empty string.
- The **& (AND)** operator ensures that both conditions are satisfied.
- The resulting DataFrame contains only valid records.

## Q13: How do you use the `.agg()` function to calculate multiple statistics at once, such as the **min**, **max**, and **mean** of the `Sales` column?

### Answer

The **`.agg()`** function is used to perform **multiple aggregate operations** on one or more columns in a single query. It improves readability and avoids writing multiple aggregation statements.

The following example calculates the **minimum**, **maximum**, and **average (mean)** values of the **Sales** column.

In [27]:
# Calculate multiple statistics using agg()
sales_statistics = df.agg(
    min("Sales").alias("Minimum Sales"),
    max("Sales").alias("Maximum Sales"),
    avg("Sales").alias("Average Sales")
)

# Display the result
sales_statistics.show()

+-------------+-------------+-----------------+
|Minimum Sales|Maximum Sales|    Average Sales|
+-------------+-------------+-----------------+
|        0.556|    11199.968|223.5581892444259|
+-------------+-------------+-----------------+



### Explanation

- **min("Sales")** calculates the minimum sales value.
- **max("Sales")** calculates the maximum sales value.
- **avg("Sales")** calculates the average (mean) sales value.
- **alias()** renames the output columns for better readability.
- **agg()** allows multiple aggregate functions to be executed in a single operation.

### Sample Output

| Minimum Sales | Maximum Sales | Average Sales |
|---------------|--------------:|--------------:|
|        0.556|    11199.968|223.5581892444259|

> **Note:** The actual values may vary depending on your Superstore dataset.

## Q14: In the context of cleaning a dataset, what is the risk of using `inferSchema=True` when your source data contains messy or inconsistent date formats?

### Answer

The **`inferSchema=True`** option automatically detects the data type of each column while reading a dataset. However, if the source data contains **messy or inconsistent date formats**, Spark may infer the wrong data type or fail to parse some values correctly.

### Risks

- **Incorrect Data Type:** Spark may treat the date column as a **StringType** instead of a **DateType**.
- **Null Values:** Invalid or inconsistent date formats may be converted to **null** values.
- **Incorrect Analysis:** Date-based operations such as filtering, sorting, and aggregation may produce incorrect results.
- **Data Cleaning Required:** The data must be cleaned and converted to a consistent date format before analysis.

### Example

Suppose a dataset contains the following date values:

| Order Date |
|------------|
| 2025-07-01 |
| 01/07/2025 |
| July 1, 2025 |
| 2025/07/01 |

Since the date formats are different, Spark may not correctly infer the column as a **DateType**. Some values may remain as strings or become **null**.

### Conclusion

When working with inconsistent date formats, it is better to read the column as a **StringType**, clean the data, and then explicitly convert it to **DateType** using functions like **`to_date()`** or **`to_timestamp()`**.

## Q15: Write a final processing pipeline that:

- Filters out duplicates.
- Fills null prices with `0`.
- Groups by `Product ID` to calculate total revenue.

### Answer

**Note:** The Superstore dataset does not contain the columns **store_id** and **price**. Therefore, **Product ID** is used as the store identifier and **Sales** is used as the price column.

In [29]:

# Data Processing Pipeline
result = (
    df
    .dropDuplicates()                          # Remove duplicate rows
    .na.fill({"Sales": 0})                     # Fill null values in Sales with 0
    .groupBy("Product ID")                     # Group by Product ID
    .agg(sum("Sales").alias("Total Revenue"))  # Calculate total revenue
)

# Display the result
result.show()

+---------------+------------------+
|     Product ID|     Total Revenue|
+---------------+------------------+
|OFF-AR-10003504|           110.424|
|FUR-FU-10000409| 98.86800000000001|
|TEC-MA-10001047|           9099.93|
|OFF-EN-10002600|             61.95|
|OFF-PA-10004735|            136.08|
|OFF-AP-10002495|            285.43|
|FUR-CH-10003833|           682.976|
|OFF-PA-10002615|               0.0|
|FUR-CH-10000863|2476.0719999999997|
|FUR-TA-10004086|          1212.318|
|OFF-PA-10000575|           184.644|
|OFF-PA-10001457|186.49599999999998|
|OFF-LA-10004008|              5.76|
|OFF-PA-10004243|            265.58|
|OFF-PA-10001639|            92.016|
|OFF-BI-10001636|            40.464|
|FUR-TA-10000849|            145.98|
|TEC-PH-10001527|             343.6|
|TEC-PH-10002885| 7538.028000000001|
|FUR-TA-10001705|            1488.2|
+---------------+------------------+
only showing top 20 rows


### Explanation

1. **dropDuplicates()** removes duplicate records from the DataFrame.
2. **na.fill({"Sales": 0})** replaces null values in the **Sales** column with **0**.
3. **groupBy("Product ID")** groups all records by Product ID.
4. **sum("Sales")** calculates the total revenue for each product.
5. **alias("Total Revenue")** renames the output column for better readability.

In [6]:
df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|  148|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+

